In [1]:
import numpy as np
import optimistix as optx
from atmodeller import (
    EquilibriumModel,
    Planet,
    ChemicalSpecies,
    SpeciesNetwork,
    debug_logger,
    earth_oceans_to_hydrogen_mass,
    SolverParameters,
)
from atmodeller.eos import get_eos_models
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer
from atmodeller.interfaces import FugacityConstraintProtocol, ThermodynamicStateProtocol
from atmodeller import debug_logger
from atmodeller.classes import EquilibriumModel
from atmodeller.containers import (
    ChemicalSpecies,
    FixedFugacityConstraint,
    SpeciesNetwork,
    ThermodynamicState,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

from scipy.interpolate import interp1d
from scipy.optimize import fsolve, root_scalar
import os

from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
formatter.set_useOffset(False)
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

logger = debug_logger()

Atmodeller initialized with double precision (float64)


In [9]:
def peridotite_solidus(P_bar): #On the cooling of a deep terrestrial magma ocean
    #P_bar: pressure in bar
    #Returns: solidus temperature in K
    P_Pa = P_bar*10**5
    T_sol = 1661.2*((P_Pa/(1.336*10**9))+1)**(1/7.437)
    return T_sol

def peridotite_liquidus(P_bar):
    #P_bar: pressure in bar
    #Returns: liquidus temperature in K
    P_Pa = P_bar*10**5
    T_liq = 1982.1*((P_Pa/(6.594*10**9))+1)**(1/5.374)
    
    return T_liq

def give_melt(Pressure, Temperature):
    #Give pressure in bar and Temperature in Kelvin
    #Returns: if magma ocean is possible or not
    Tline = peridotite_solidus(Pressure)
    if Tline <= Temperature:
        return 1
    elif Tline> Temperature:
        return 0

def create_melt_list(excel_file):
    #Excel file from atmodeller to generate a list with values for 0 or 1 given melt = 1 or 0
    mantle_melt = []
    Pressures = pd.read_excel(excel_file, sheet_name="state")["pressure"]
    Temperatures = pd.read_excel(excel_file, sheet_name="state")["temperature"]

    for i in range(len(Pressures)):
        mantle_melt.append(give_melt(Pressures[i], Temperatures[i]))


    return mantle_melt

def create_melt(Pressures, Temperatures):
    #Excel file from Atmodeller to generate a list with values for 0 or 1
    mantle_melt = []
    for i in range(len(Pressures)):
        mantle_melt.append(give_melt(Pressures[i], Temperatures[i]))


    return mantle_melt

In [2]:
SiO2_mantlemasspercent_palme14: float = 45.4
total_mantlemasspercent_palme14: float = 98.41
core_mass_fraction: float = 0.327  # typical values used are between 0.325 - 0.33
Si_massfraction_palme14: float = round(
    SiO2_mantlemasspercent_palme14
    / total_mantlemasspercent_palme14
    * (1 - core_mass_fraction)
    / (28.0855 + 2 * 15.999)
    * 28.0855,
    3,
)

O_massfraction_palme14: float = Si_massfraction_palme14 / 28.0855 * 2 * 15.999
# Lodders et al. (2009) Springer Book Chapter - Table 8 (in wt%)

H_masspercent_lodders09: float = 73.9
He_masspercent_lodders09: float = 24.69
C_masspercent_lodders09: float = 0.22
N_masspercent_lodders09: float = 0.07
O_masspercent_lodders09: float = 0.63
Si_masspercent_lodders09: float = 0.07
Mg_masspercent_lodders09: float = 0.06

# Lodders et al. (2009) Springer Book Chapter - Table 6 (log-normalalized abundances relative to H)

H_logN: float = 12
He_logN: float = 10.93
C_logN: float = 8.39
N_logN: float = 7.86
O_logN: float = 8.73
Si_logN: float = 7.53
Mg_logN: float = 7.54 

In [3]:
eos_models = get_eos_models()
sol_models = get_solubility_models()

H2O_g = ChemicalSpecies.create_gas("H2O")
H2O_gs = ChemicalSpecies.create_gas("H2O", solubility=sol_models["H2O_peridotite_sossi23"])
H2O_rgs = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
    solubility=sol_models["H2O_peridotite_sossi23"],
)
H2O_rg = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
)

H2_g = ChemicalSpecies.create_gas("H2")
H2_gs = ChemicalSpecies.create_gas("H2", solubility=sol_models["H2_basalt_hirschmann12"])
H2_rgs = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"], solubility=sol_models["H2_basalt_hirschmann12"]
)
H2_rg = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"]
)

O2_g = ChemicalSpecies.create_gas("O2")
O2_rg = ChemicalSpecies.create_gas("O2", activity=eos_models["O2_cs_shi92"])

OSi_g = ChemicalSpecies.create_gas("OSi")
OSi_rg = ChemicalSpecies.create_gas("OSi", activity=eos_models["OSi_rk49_connolly16"])

H4Si_g = ChemicalSpecies.create_gas("H4Si")
H4Si_rg = ChemicalSpecies.create_gas("H4Si", activity=eos_models["H4Si_wang18"])

O2Si_cd = ChemicalSpecies.create_condensed("O2Si", state="cd")

C_cd = ChemicalSpecies.create_condensed("C", state="cd")
CSi_cd = ChemicalSpecies.create_condensed("CSi", state="cd")
Si_cd = ChemicalSpecies.create_condensed("Si", state="cd")
N4Si3_cd = ChemicalSpecies.create_condensed("N4Si3", state="cd")
H2O_cd = ChemicalSpecies.create_condensed("H2O", state="cd")

CO2_g = ChemicalSpecies.create_gas("CO2")
CO2_gs = ChemicalSpecies.create_gas("CO2", solubility=sol_models["CO2_basalt_dixon95"])
CO2_rgs = ChemicalSpecies.create_gas(
    "CO2", solubility=sol_models["CO2_basalt_dixon95"], activity=eos_models["CO2_cs_shi92"]
)
CO2_rg = ChemicalSpecies.create_gas(
    "CO2",activity=eos_models["CO2_cs_shi92"]
)

CO_g = ChemicalSpecies.create_gas("CO")
CO_gs = ChemicalSpecies.create_gas("CO", solubility=sol_models["CO_basalt_yoshioka19"])
CO_rgs = ChemicalSpecies.create_gas(
    "CO", solubility=sol_models["CO_basalt_yoshioka19"], activity=eos_models["CO_cs_shi92"]
)
CO_rg = ChemicalSpecies.create_gas(
    "CO", activity=eos_models["CO_cs_shi92"]
)

CH4_g = ChemicalSpecies.create_gas("CH4")
CH4_gs = ChemicalSpecies.create_gas("CH4", solubility=sol_models["CH4_basalt_ardia13"])
CH4_rgs = ChemicalSpecies.create_gas(
    "CH4", solubility=sol_models["CH4_basalt_ardia13"], activity=eos_models["CH4_cs_shi92"]
)
CH4_rg = ChemicalSpecies.create_gas(
    "CH4", activity=eos_models["CH4_cs_shi92"]
)

N2_g = ChemicalSpecies.create_gas("N2")
N2_gs = ChemicalSpecies.create_gas("N2", solubility=sol_models["N2_basalt_libourel03"])
N2_rgs = ChemicalSpecies.create_gas(
    "N2",solubility=sol_models["N2_basalt_libourel03"], activity=eos_models["N2_cs_saxena87"]
)
N2_rg = ChemicalSpecies.create_gas(
    "N2", activity=eos_models["N2_cs_saxena87"]
)


NH3_g = ChemicalSpecies.create_gas("H3N")
NH3_rg = ChemicalSpecies.create_gas("H3N", activity=eos_models["H3N_rk49_reid87"])

He_g = ChemicalSpecies.create_gas("He")
He_gs = ChemicalSpecies.create_gas("He", solubility=sol_models["He_basalt_jambon86"])
He_rgs = ChemicalSpecies.create_gas(
    "He", solubility=sol_models["He_basalt_jambon86"], activity=eos_models["He_chabrier21"]
)
He_rg = ChemicalSpecies.create_gas(
    "He", activity=eos_models["He_chabrier21"]
)


species_HOSi_magma_nosol_ideal = SpeciesNetwork((H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_ideal = SpeciesNetwork((H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_real = SpeciesNetwork((H2O_rgs, H2_rgs, O2_rg, OSi_rg, H4Si_rg, O2Si_cd))
species_HOSi_atm_sol_real = SpeciesNetwork((H2O_g, H2_g, O2_g, H2O_cd,OSi_rg, H4Si_rg))


species_HHeCNOSi_magma_nosol_ideal = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
)
species_HHeCNOSi_magma_sol_ideal = SpeciesNetwork(
    (H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_gs, CO_gs, CH4_gs, N2_gs, NH3_g, He_gs)
)
species_HHeCNOSi_magma_sol_real = SpeciesNetwork(
    (
        H2O_rgs,
        H2_rgs,
        O2_rg,
        OSi_rg,
        H4Si_rg,
        O2Si_cd,
        CO2_rgs,
        CO_rgs,
        CH4_rgs,
        N2_rgs,
        NH3_rg,
        He_rgs,
    )
)

species_HHeCNOSi_atmosphere = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg, H2O_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
     CO2_rg, CO_rg, CH4_rg, N2_rg, NH3_rg, He_rgs)
)
species_HHeCNOSi_atmosphere = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
     CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
)

In [5]:
number_of_realisations = 50

# Overall structure 
R_earth = 6.371e6
R_uranus = 3.939 * R_earth
M_earth = 5.972e24 #kg
M_uranus = 14.536 * M_earth

#fO2_log10_shifts = [-1]
fO2_log10_shifts = np.linspace(-6, 0, num=30)  # IW-6 to IW
fO2_log10_shifts = [-6,-5,-4,-3,-2,-1,0]
hmps = np.logspace(1.2,1.2, num=1)  # wt% H
h_kgs = hmps / 100 * M_uranus  # kg

si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09


In [15]:
filename = f"/home/mathias/Margtest/Magrathea/result_final0/results.xlsx"
#here should be a different filename

params = pd.read_excel(filename, sheet_name="Run_Parameters")
mask = params["mass_of_hydro"] == 0
Radii = pd.read_excel(filename, sheet_name="Results")["Radius (earth)"][mask]
Masses = pd.read_excel(filename, sheet_name="Results")["M (earth)"][mask]
Temperatures_layer = pd.read_excel(filename, sheet_name="Results")["T (K)"][mask]
Pressures_layer = pd.read_excel(filename, sheet_name="Results")["P (GPa)"][mask]

Radii = Radii.iloc[::-1].reset_index(drop=True)
Masses = Masses.iloc[::-1].reset_index(drop=True)
Temperatures_layer = Temperatures_layer.iloc[::-1].reset_index(drop=True)
Pressures_layer = Pressures_layer.iloc[::-1].reset_index(drop=True)
Pressures_layer = np.array(Pressures_layer)*1e4
#This pressure is times 1e4 to have from GPa to bar 

#Radii = Radii.iloc[125:].reset_index(drop=True)
#Masses = Masses.iloc[125:].reset_index(drop=True)
#Temperatures_layer = Temperatures_layer.iloc[125:].reset_index(drop=True)
#Pressures_layer = Pressures_layer[125:]

In [16]:
#This one works

init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1
model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)
rho_centre = 3000
fO2_log10_shift = -4
hmps = np.logspace(1,1, num=1)  # trying one hmps to see
for i in range(len(Radii)):
    radius = Radii[i]*R_earth
    masses = Masses[i]*M_earth
    temp_layer = Temperatures_layer[i]
    pressure_layer = Pressures_layer[i]
    shift = f"{fO2_log10_shift:.2g}"
    #Based on layer pressure and magrathea
    h_kgs = hmps / 100 * M_uranus  # kg

    si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
    mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
    o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
    c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
    n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
    he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09
    si_kg_magma = Si_massfraction_palme14 * masses
    for init_metallicity in init_metallicitys:
        mass_constraints = {
        "H": h_kgs,
        "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
    }
        state_gas: ThermodynamicStateProtocol = ThermodynamicState(temp_layer, pressure_layer,mass=5e22, melt_fraction=give_melt(temp_layer,pressure_layer))
        # Magma - No solubility - Ideal Gas
        model_magma_nosol_ideal.solve(
                state=state_gas,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
                solver_parameters=SolverParameters(multistart=20)
            )
        output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
        output_magma_nosol_ideal.to_excel(
            f"./TEST"
        )
        output_magma_nosol_ideal.to_pickle(
            f"./TEST"
        )
    
        # Magma - Soluility - Ideal Gas
        initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
        model_magma_sol_ideal.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_ideal = model_magma_sol_ideal.output
        # output_magma_sol_ideal.quick_look()
        output_magma_sol_ideal.to_excel(
            f"./TEST"
        )
        output_magma_sol_ideal.to_pickle(
            f"./TEST"
        )
        # Magma - Solubility - Real Gas
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        model_magma_sol_real.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=50)
        )
        output_magma_sol_real = model_magma_sol_real.output
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        output_magma_sol_real.quick_look()
        output_magma_sol_real.to_excel(
            f"./TEST1"
        )
        output_magma_sol_real.to_pickle(
            f"./TEST1"
        )

[19:25:30 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[19:25:30 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[19:25:30 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[19:25:30 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[19:25:30 - atmodeller.classes             - INFO     ] - Thermodynamic

7.17876204e+25
2617.2794
787389.89


[19:25:40 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[19:25:40 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[19:25:40 - atmodeller.classes             - INFO     ] - Solver steps (max) = 44
[19:25:40 - atmodeller.output_core         - DEBUG    ] - Creating Output
[19:25:40 - atmodeller.output              - INFO     ] - Writing output to excel
[19:25:40 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[19:25:40 - atmodeller.output_core         - INFO     ] - Computing asdict output
[19:25:41 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[19:25:41 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[19:25:41 - atmodeller.output              - INFO     ] - Writing output to pickle
[19:25:41 - atmodeller.output_core         - DEBUG    ] - Returning cached to_

7.176313520000001e+25
2611.5222
788319.02


[19:27:23 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[19:27:23 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[19:27:23 - atmodeller.output              - INFO     ] - Writing output to pickle
[19:27:23 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[19:27:23 - atmodeller.output              - INFO     ] - Output written to TEST.pkl
[19:27:23 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[19:27:23 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[19:27:23 - atmodeller.classes             - INFO     ] - Solver steps (max) = 6
[19:27:23 - atmodeller.output_core         - DEBUG    ] - Creating Output
[19:27:23 - atmodeller.output              - INFO     ] - Writing output to excel
[19:27:23 - atmodeller.output_core         - INFO     ] - Computing t

7.173924720000001e+25
2605.7658
789278.9500000001


... (16 outputs left out)

[19:28:05 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[19:28:05 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[19:28:05 - atmodeller.output              - INFO     ] - Writing output to pickle
[19:28:05 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[19:28:05 - atmodeller.output              - INFO     ] - Output written to TEST.pkl
[19:28:05 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[19:28:05 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[19:28:05 - atmodeller.classes             - INFO     ] - Solver steps (max) = 5
[19:28:05 - atmodeller.output_core         - DEBUG    ] - Creating Output
[19:28:05 - atmodeller.output              - INFO     ] - Writing output to excel
[19:28:05 - atmodeller.output_core         - INFO     ] - Computing t

7.151231120000001e+25
2552.0536
798234.91


[19:28:10 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[19:28:10 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[19:28:10 - atmodeller.output              - INFO     ] - Writing output to pickle
[19:28:10 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[19:28:10 - atmodeller.output              - INFO     ] - Output written to TEST.pkl
[19:28:10 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[19:28:10 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[19:28:10 - atmodeller.classes             - INFO     ] - Solver steps (max) = 5
[19:28:10 - atmodeller.output_core         - DEBUG    ] - Creating Output
[19:28:10 - atmodeller.output              - INFO     ] - Writing output to excel
[19:28:10 - atmodeller.output_core         - INFO     ] - Computing t

OverflowError: cannot convert float infinity to integer

In [17]:
#This one does not
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1
model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)
rho_centre = 3000
fO2_log10_shift = -4
hmps = np.logspace(1,1, num=1)  # trying one hmps to see
for i in range(len(Radii)):
    radius = Radii[i]*R_earth
    masses = Masses[i]*M_earth
    temp_layer = Temperatures_layer[i]
    pressure_layer = Pressures_layer[i]
    shift = f"{fO2_log10_shift:.2g}"
    #Based on layer pressure and magrathea
    h_kgs = hmps / 100 * M_uranus  # kg

    si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
    mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
    o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
    c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
    n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
    he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09
    si_kg_magma = Si_massfraction_palme14 * masses
    for init_metallicity in init_metallicitys:
        mass_constraints = {
        "H": h_kgs,
        "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
    }
        state_gas: ThermodynamicStateProtocol = ThermodynamicState(temp_layer, pressure_layer,mass=masses, melt_fraction=give_melt(temp_layer,pressure_layer))
        # Magma - No solubility - Ideal Gas
        model_magma_nosol_ideal.solve(
                state=state_gas,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
                solver_parameters=SolverParameters(multistart=20)
            )
        output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
        output_magma_nosol_ideal.to_excel(
            f"./TEST"
        )
        output_magma_nosol_ideal.to_pickle(
            f"./TEST"
        )
    
        # Magma - Soluility - Ideal Gas
        initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
        model_magma_sol_ideal.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=20)
        )
        output_magma_sol_ideal = model_magma_sol_ideal.output
        # output_magma_sol_ideal.quick_look()
        output_magma_sol_ideal.to_excel(
            f"./TEST"
        )
        output_magma_sol_ideal.to_pickle(
            f"./TEST"
        )
        # Magma - Solubility - Real Gas
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        model_magma_sol_real.solve(
            state=state_gas,
            mass_constraints=mass_constraints,
            initial_log_number_moles=initial_log_number_moles,
            fugacity_constraints={O2_rg.name: IronWustiteBuffer(fO2_log10_shift)},
            solver_parameters=SolverParameters(multistart=50)
        )
        output_magma_sol_real = model_magma_sol_real.output
        initial_log_number_moles = output_magma_sol_ideal.log_number_moles
        output_magma_sol_real.quick_look()
        output_magma_sol_real.to_excel(
            f"./TEST1"
        )
        output_magma_sol_real.to_pickle(
            f"./TEST1"
        )

[19:28:38 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[19:28:38 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[19:28:38 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[19:28:38 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[19:28:38 - atmodeller.classes             - INFO     ] - Thermodynamic

OverflowError: cannot convert float infinity to integer